# Libraries

In [12]:
import pandas as pd
import numpy as np

from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.utils.class_weight import compute_sample_weight

# Load Data

In [13]:
train = pd.read_excel("./Dataset/train.xlsx")
test = pd.read_excel("./Dataset/student_test.xlsx")

# Cleaning Functions

In [14]:
def clean_text(x):
    if pd.isna(x):
        return np.nan
    return str(x).strip().lower()


def clean_binary(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    if x in ["yes", "true", "y", "1"]:
        return 1
    if x in ["no", "false", "n", "0"]:
        return 0
    return np.nan


def to_numeric_safe(df, cols):
    for col in cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

# Utility Functions

In [15]:
def optimize_threshold(y_true, probs, start=0.10, stop=0.90, step=0.01):
    best_f1 = -1
    best_t = 0.5

    for t in np.arange(start, stop, step):
        preds = (probs >= t).astype(int)
        f1 = f1_score(y_true, preds)
        if f1 > best_f1:
            best_f1 = f1
            best_t = float(t)

    return best_t, best_f1


def evaluate_model(name, model, X_train, y_train, X_val, y_val, fit_params=None):
    print("\n" + "=" * 70)
    print(f"Training: {name}")
    print("=" * 70)

    fit_params = fit_params or {}
    model.fit(X_train, y_train, **fit_params)

    val_probs = model.predict_proba(X_val)[:, 1]
    best_t, best_f1 = optimize_threshold(y_val, val_probs)
    val_preds = (val_probs >= best_t).astype(int)

    metrics = {
        "accuracy": accuracy_score(y_val, val_preds),
        "precision": precision_score(y_val, val_preds, zero_division=0),
        "recall": recall_score(y_val, val_preds, zero_division=0),
        "f1": f1_score(y_val, val_preds, zero_division=0),
        "roc_auc": roc_auc_score(y_val, val_probs),
        "avg_precision": average_precision_score(y_val, val_probs),
    }

    print(f"\n{name} | Best threshold: {best_t:.2f}")
    print(f"{name} | Metrics:")
    print(f"  Accuracy          : {metrics['accuracy']:.6f}")
    print(f"  Precision         : {metrics['precision']:.6f}")
    print(f"  Recall            : {metrics['recall']:.6f}")
    print(f"  F1                : {metrics['f1']:.6f}")
    print(f"  ROC-AUC           : {metrics['roc_auc']:.6f}")
    print(f"  Average Precision : {metrics['avg_precision']:.6f}")
    print(f"\n{name} | Confusion Matrix:\n{confusion_matrix(y_val, val_preds)}")
    print(f"\n{name} | Classification Report:\n{classification_report(y_val, val_preds, zero_division=0)}")

    return {
        "name": name,
        "model": model,
        "threshold": best_t,
        **metrics,
    }

# Clean Data

In [16]:
numeric_cols = [
    "transaction_amount",
    "avg_transaction_amount_7d",
    "transaction_frequency_24h",
    "failed_transaction_count_24h",
    "account_age_days",
]

train = to_numeric_safe(train, numeric_cols)
test = to_numeric_safe(test, numeric_cols)

binary_cols = [
    "is_international",
    "unusual_amount_flag",
    "multiple_transactions_short_time",
    "high_risk_device_flag",
]

for col in binary_cols:
    train[col] = train[col].apply(clean_binary)
    test[col] = test[col].apply(clean_binary)

for df in [train, test]:
    df["transaction_timestamp"] = pd.to_datetime(df["transaction_timestamp"], errors="coerce")

    df["hour"] = df["transaction_timestamp"].dt.hour
    df["day"] = df["transaction_timestamp"].dt.day
    df["month"] = df["transaction_timestamp"].dt.month
    df["dayofweek"] = df["transaction_timestamp"].dt.dayofweek

    df.drop("transaction_timestamp", axis=1, inplace=True)

cat_cols = ["payment_method", "device_type", "location", "merchant_category"]

for df in [train, test]:
    for col in cat_cols:
        df[col] = df[col].apply(clean_text).fillna("missing")


# Feature Engineering

In [17]:
for df in [train, test]:
    df["amount_to_avg_ratio"] = df["transaction_amount"] / (df["avg_transaction_amount_7d"] + 1e-6)
    df["amount_diff"] = df["transaction_amount"] - df["avg_transaction_amount_7d"]

    df["risk_score"] = (
        df["is_international"].fillna(0) * 2
        + df["unusual_amount_flag"].fillna(0) * 2
        + df["multiple_transactions_short_time"].fillna(0) * 1.5
        + df["high_risk_device_flag"].fillna(0) * 2
    )

    df["missing_values_count"] = df.isna().sum(axis=1)

# Split Data

In [18]:
X = train.drop(["label", "transaction_id"], axis=1)
y = train["label"]

X_test = test.drop(["id", "transaction_id"], axis=1)

categorical_cols = [
    "customer_id",
    "payment_method",
    "device_type",
    "location",
    "merchant_category",
]

for col in categorical_cols:
    X[col] = X[col].astype(str)
    X_test[col] = X_test[col].astype(str)

cat_features = [X.columns.get_loc(col) for col in categorical_cols]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# Define Models

In [19]:
numeric_features = [col for col in X.columns if col not in categorical_cols]

sklearn_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
                    (
                        "encoder",
                        OrdinalEncoder(
                            handle_unknown="use_encoded_value",
                            unknown_value=-1,
                        ),
                    ),
                ]
            ),
            categorical_cols,
        ),
    ],
    remainder="drop",
)

sample_weight_train = compute_sample_weight(class_weight="balanced", y=y_train)

catboost_model = CatBoostClassifier(
    iterations=1500,
    depth=8,
    learning_rate=0.03,
    loss_function="Logloss",
    eval_metric="F1",
    random_seed=42,
    class_weights=[1, 6],
    verbose=200,
)

random_forest_model = Pipeline(
    steps=[
        ("prep", sklearn_preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=600,
                max_depth=None,
                min_samples_split=4,
                min_samples_leaf=2,
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

hist_gradient_model = Pipeline(
    steps=[
        ("prep", sklearn_preprocessor),
        (
            "model",
            HistGradientBoostingClassifier(
                learning_rate=0.04,
                max_iter=600,
                max_leaf_nodes=31,
                l2_regularization=0.05,
                early_stopping=True,
                random_state=42,
            ),
        ),
    ]
)

# Train & Compare Models


In [20]:
results = []

results.append(
    evaluate_model(
        "CatBoost",
        catboost_model,
        X_train,
        y_train,
        X_val,
        y_val,
        fit_params={
            "cat_features": cat_features,
            "eval_set": (X_val, y_val),
            "use_best_model": True,
        },
    )
)

results.append(
    evaluate_model(
        "RandomForest",
        random_forest_model,
        X_train,
        y_train,
        X_val,
        y_val,
    )
)

results.append(
    evaluate_model(
        "HistGradientBoosting",
        hist_gradient_model,
        X_train,
        y_train,
        X_val,
        y_val,
        fit_params={"model__sample_weight": sample_weight_train},
    )
)

comparison = pd.DataFrame(
    [
        {
            "model": r["name"],
            "best_threshold": r["threshold"],
            "accuracy": r["accuracy"],
            "precision": r["precision"],
            "recall": r["recall"],
            "f1": r["f1"],
            "roc_auc": r["roc_auc"],
            "avg_precision": r["avg_precision"],
        }
        for r in results
    ]
).sort_values("f1", ascending=False)

print("\n")
print("MODEL COMPARISON - sorted by validation F1")
print("#" * 90)
print(comparison.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

best = max(results, key=lambda x: x["f1"])
print("\n")
print(f"Best model: {best['name']}")
print(f"Best validation F1: {best['f1']:.6f}")
print(f"Best threshold: {best['threshold']:.2f}")


Training: CatBoost
0:	learn: 0.9465330	test: 0.9460828	best: 0.9460828 (0)	total: 85.6ms	remaining: 2m 8s
200:	learn: 0.9635547	test: 0.9640447	best: 0.9641760 (92)	total: 32.5s	remaining: 3m 29s
400:	learn: 0.9639128	test: 0.9642891	best: 0.9642891 (372)	total: 1m 4s	remaining: 2m 57s
600:	learn: 0.9642699	test: 0.9641385	best: 0.9643400 (459)	total: 1m 38s	remaining: 2m 28s
800:	learn: 0.9644455	test: 0.9639924	best: 0.9643400 (459)	total: 2m 15s	remaining: 1m 57s
1000:	learn: 0.9646238	test: 0.9639675	best: 0.9643400 (459)	total: 2m 51s	remaining: 1m 25s
1200:	learn: 0.9647818	test: 0.9638213	best: 0.9643400 (459)	total: 3m 41s	remaining: 55.1s
1400:	learn: 0.9649576	test: 0.9636750	best: 0.9643400 (459)	total: 4m 52s	remaining: 20.6s
1499:	learn: 0.9650443	test: 0.9635889	best: 0.9643400 (459)	total: 5m 12s	remaining: 0us

bestTest = 0.9643400372
bestIteration = 459

Shrink model to first 460 iterations.

CatBoost | Best threshold: 0.78
CatBoost | Metrics:
  Accuracy          : 0.

# Train With Best Model

In [21]:
if best["name"] == "CatBoost":
    final_model = CatBoostClassifier(
        iterations=1500,
        depth=8,
        learning_rate=0.03,
        loss_function="Logloss",
        eval_metric="F1",
        random_seed=42,
        class_weights=[1, 6],
        verbose=200,
    )
    final_model.fit(X, y, cat_features=cat_features)

elif best["name"] == "RandomForest":
    final_model = random_forest_model
    final_model.fit(X, y)

else:
    final_model = hist_gradient_model
    sample_weight_full = compute_sample_weight(class_weight="balanced", y=y)
    final_model.fit(X, y, model__sample_weight=sample_weight_full)

0:	learn: 0.9565484	total: 270ms	remaining: 6m 44s
200:	learn: 0.9637581	total: 42.8s	remaining: 4m 36s
400:	learn: 0.9640170	total: 1m 23s	remaining: 3m 49s
600:	learn: 0.9641522	total: 2m 2s	remaining: 3m 3s
800:	learn: 0.9642439	total: 2m 44s	remaining: 2m 23s
1000:	learn: 0.9643375	total: 3m 26s	remaining: 1m 42s
1200:	learn: 0.9644658	total: 4m 10s	remaining: 1m 2s
1400:	learn: 0.9646329	total: 5m 31s	remaining: 23.4s
1499:	learn: 0.9647124	total: 6m 11s	remaining: 0us


# Gererate Output

In [22]:
test_probs = final_model.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= best["threshold"]).astype(int)

submission = pd.DataFrame(
    {
        "id": test["id"],
        "label": test_preds,
    }
)

submission.to_csv("submission.csv", index=False)

print("\n submission.csv created successfully using:", best["name"])



 submission.csv created successfully using: CatBoost
